In [ ]:
pip install transformers torch scikit-learn umap-learn matplotlib seaborn

In [ ]:
import torch
from transformers import BertTokenizer, BertModel
import umap.umap_ as umap
import plotly.express as px
import pandas as pd
import numpy as np

In [ ]:
#Inizializzazione BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased', output_hidden_states=True)
model.eval()

In [ ]:
corpus = {
    # --- Bassa Complessità (Monosemia / Bassa Entropia) ---
    "cat": [
        "My cat loves to sleep on the sofa.",
        "The stray cat caught a mouse.",
        "Don't forget to feed the cat.",
        "The black cat crossed the street.",
        "She adopted a rescue cat from the shelter."
    ],
    "school": [ # H(w) = 0.29
        "The children are learning math at school.",
        "High school students are preparing for exams.",
        "The school board approved the new budget.",
        "He walked his daughter to school.",
        "A large school of fish swam past the diver."
    ],

    # --- Media Complessità (Omonimia e Polisemia Regolare) ---
    "bass": [ # Omonimia netta
        "He plays the bass guitar in a rock band.",
        "The bass was turned up so loud the speakers shook.",
        "We caught a large sea bass on our fishing trip.",
        "Freshwater bass is delicious when grilled.",
        "She has a deep bass voice."
    ],
    "chicken": [ # Pattern produttivo Animale-Carne
        "The farmer feeds the chicken every morning.",
        "A fox sneaked in and stole a chicken.",
        "We are having roast chicken for dinner.",
        "This chicken soup is very comforting.",
        "Don't be a chicken, just jump into the water!"
    ],
    "book": [ # Dot objects (Fisico vs Contenuto)
        "The heavy book fell off the shelf.",
        "She spilled coffee on the cover of the book.",
        "I read a fascinating book about history.",
        "He is writing a new book.",
        "Did you book the flights for our holiday?"
    ],

    # --- Alta Complessità (Polisemia Genuina e Continuum) ---
    "bank": [ # H(w) = 0.99
        "I deposited my money in the bank today.",
        "The central bank raised interest rates.",
        "She sat on the river bank and watched the water.",
        "The boat was moored on the muddy bank.",
        "The plane performed a steep bank to the left."
    ],
    "paper": [ # H(w) = 1.30 (Continuum semantico)
        "I need a blank sheet of paper to draw on.",
        "The paper is made from recycled materials.",
        "She published a groundbreaking scientific paper.",
        "I read about the scandal in the morning paper.",
        "Please sign this piece of paper."
    ],
    "head": [ # Catene radiali (struttura cognitiva)
        "He wore a hat on his head.",
        "Hit the nail on the head.",
        "She sits at the head of the table.",
        "He was appointed head of the department.",
        "The beer had a thick white head."
    ],
    "match": [
        "We watched a thrilling football match on TV.",
        "She won the tennis match in straight sets.",
        "He struck a match to light the campfire.",
        "Do you have a match for this candle?",
        "Your socks don't match your shoes."
    ],
    "run": [ # H(w) = 2.15 (Complessità estrema)
        "I go for a run in the park every morning.",
        "She knows how to run a successful business.",
        "The river will run dry if it doesn't rain soon.",
        "You have a run in your stockings.",
        "The program will run automatically on startup."
    ]
}

In [ ]:
def get_embedding(sentence, target):
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        # Usiamo il penultimo layer per una migliore resa semantica
        hidden_states = outputs.hidden_states[-2]

    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    try:
        target_idx = tokens.index(target.lower())
        return hidden_states[0][target_idx].numpy()
    except:
        return None

In [ ]:
#Estraiamo gli embeddings
data = []
for word, sentences in corpus.items():
    for sent in sentences:
        emb = get_embedding(sent, word)
        if emb is not None:
            data.append({'word': word, 'sentence': sent, 'embedding': emb})

df_data = pd.DataFrame(data)
embeddings_matrix = np.stack(df_data['embedding'].values)

In [ ]:
# utiliziamo UMAP per il bilanciamento
reducer = umap.UMAP(n_neighbors=5, min_dist=0.3, random_state=42)
projections = reducer.fit_transform(embeddings_matrix)
df_data['x'], df_data['y'] = projections[:, 0], projections[:, 1]

In [ ]:
fig = px.scatter(
    df_data, x='x', y='y', color='word',
    hover_data={'sentence': True, 'x': False, 'y': False},
    title="Esplorazione Interattiva della Polisemia (BERT + UMAP)",
    labels={'word': 'Parola Target'}
)

fig.update_traces(marker=dict(size=12, opacity=0.8, line=dict(width=1, color='white')))
fig.show()